# 05 — HOPE full model

End-to-end walkthrough of `hope.model.HOPE`.

Paper references:

- §8.1, Eq. 76-91 — Self-Referential Titans (the self-modifying block).
- §7.1, Eq. 70-71 — Continuum Memory System.
- §8 "Hope-Attention" paragraph — global softmax attention.
- §8.3, Figure 5 — HOPE combines all three.

Stack used here: token + positional embedding → SelfModifyingLayer → CMS → HopeAttention → LayerNorm → LM head.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from hope.model import HOPE

tf.random.set_seed(0)
np.random.seed(0)


## Build a tiny HOPE and sanity-check the forward shape

In [ ]:
model = HOPE(
    vocab_size=16,
    d_model=32,
    n_self_mod_layers=1,
    cms_banks=(1, 4),
    cms_decays=(0.01, 0.005),
    n_heads=4,
    max_seq_len=8,
)
x = tf.constant([[1, 2, 3, 4, 5, 6, 7, 8]], dtype=tf.int32)
logits = model(x)
print('logits shape:', tuple(logits.shape))


## Tiny training loop on a synthetic copy task

Predict the same token as the input. Loss should drop over a handful of Adam steps.

In [ ]:
tf.random.set_seed(1)
model = HOPE(
    vocab_size=8,
    d_model=16,
    n_self_mod_layers=1,
    cms_banks=(1, 4),
    cms_decays=(0.01, 0.005),
    n_heads=2,
    max_seq_len=4,
)
opt = tf.keras.optimizers.Adam(1e-2)
x = tf.constant([[1, 2, 3, 4]], dtype=tf.int32)
y = x

N_STEPS = 30
losses = []
for step in range(N_STEPS):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(y, logits, from_logits=True)
        )
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    losses.append(float(loss))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, marker='o')
ax.set_xlabel('step')
ax.set_ylabel('cross-entropy loss')
ax.set_title('HOPE on the synthetic copy task')
fig.tight_layout()
plt.show()

print(f'loss[0]={losses[0]:.4f}  ->  loss[-1]={losses[-1]:.4f}')
